# 🚀 Unified Kaggle AI Model Server (LLM, Embeddings, Docling Parser, & Reranker)
This notebook serves all three model services under a single FastAPI app and tunnels them to a single public Ngrok URL. It leverages **Kaggle's 2x T4 GPU Accelerator** to distribute the models:
- **GPU 0 (cuda:0)**: Qwen 2.5 Coder 7B Instruct (LLM) in 4-bit precision.
- **GPU 1 (cuda:1)**: Nomic Embedding v1.5, Docling PDF Parser, and BGE Reranker Large.

### ⚙️ Instructions
1. Turn on the **GPU Accelerator** in the notebook settings (select **GPU T4 x2**).
2. Add your **Ngrok Auth Token** to Kaggle Secrets:
   - Go to **Add-ons -> Secrets** in the top menu.
   - Add a secret named `NGROK_AUTH_TOKEN` with your token as the value.
   - Make sure the checkbox next to the secret is checked to share it with the notebook.
3. Run all cells in order.

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install -q \
    transformers \
    accelerate \
    bitsandbytes \
    torch \
    sentencepiece \
    fastapi \
    uvicorn \
    pyngrok \
    nest_asyncio \
    pydantic \
    einops \
    docling \
    python-multipart \
    pandas \
    tabulate

## 🔑 Step 2: Load Secrets and Import Libraries

In [ ]:
import os
import tempfile
import uuid
import torch
import nest_asyncio
import logging
from fastapi import FastAPI, UploadFile, Form, File, BackgroundTasks
from pydantic import BaseModel
from pyngrok import ngrok
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoModel,
    BitsAndBytesConfig
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, RapidOcrOptions
from docling.datamodel.base_models import InputFormat
from docling.chunking import HybridChunker

# Configure system-wide logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S"
)
logging.getLogger("docling").setLevel(logging.DEBUG)
logging.getLogger("docling.pipeline.base_pipeline").setLevel(logging.DEBUG)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)

PORT = 8000

# Fetch secret token from Kaggle user secrets
NGROK_AUTH_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    NGROK_AUTH_TOKEN = user_secrets.get_secret("NGROK_AUTH_TOKEN")
    print("✅ Loaded NGROK_AUTH_TOKEN from Kaggle User Secrets.")
except Exception as e:
    print("❌ Failed to load secret from Kaggle User Secrets. Checking environment variables...")
    NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN")

if not NGROK_AUTH_TOKEN:
    print("⚠️ WARNING: NGROK_AUTH_TOKEN is not set. Please add it to Add-ons -> Secrets.")

## 🤖 Step 3: Load Models onto Dual T4 GPUs

In [ ]:
print("Configuring GPU Device Distribution...")
device_llm = "cuda:0" if torch.cuda.device_count() > 0 else "cpu"
device_embedding = "cuda:1" if torch.cuda.device_count() > 1 else ("cuda:0" if torch.cuda.is_available() else "cpu")
device_reranker = "cuda:1" if torch.cuda.device_count() > 1 else ("cuda:0" if torch.cuda.is_available() else "cpu")

print(f"LLM -> {device_llm} | Embedding/Docling -> {device_embedding} | Reranker -> {device_reranker}\n")

# 1. Qwen 2.5 Coder 7B on GPU 0
print("--- Loading Qwen 2.5 Coder 7B (LLM) on GPU 0 ---")
llm_model_name = "Qwen/Qwen2.5-Coder-7B-Instruct"
llm_tokenizer = AutoTokenizer.from_pretrained(llm_model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_model_name,
    device_map={"": "cuda:0"} if torch.cuda.is_available() else "auto",
    quantization_config=bnb_config
)
llm_model.eval()
print(f"✅ Qwen model loaded successfully on device: {next(llm_model.parameters()).device}\n")

# 2. Nomic Embedding Model on GPU 1
print("--- Loading Nomic Embed Text v1.5 on GPU 1 ---")
embed_model_name = "nomic-ai/nomic-embed-text-v1.5"
embed_tokenizer = AutoTokenizer.from_pretrained(embed_model_name, trust_remote_code=True)
embed_model = AutoModel.from_pretrained(embed_model_name, trust_remote_code=True)
embed_model = embed_model.to(device_embedding)
embed_model.eval()
print(f"✅ Embedding model loaded successfully on device: {embed_model.device}\n")

# 3. BGE Reranker on GPU 1
print("--- Loading BGE Reranker Large on GPU 1 ---")
reranker_model_name = "BAAI/bge-reranker-large"
reranker_tokenizer = AutoTokenizer.from_pretrained(reranker_model_name)
reranker_model = AutoModelForSequenceClassification.from_pretrained(reranker_model_name)
reranker_model = reranker_model.to(device_reranker)
reranker_model.eval()
print(f"✅ BGE Reranker model loaded successfully on device: {reranker_model.device}\n")

# 4. Docling PDF Converter configured for GPU 1
print("--- Configuring Docling PDF Converter with GPU 1 ---")
pipeline_options = PdfPipelineOptions()
pipeline_options.accelerator_options.device = device_embedding
pipeline_options.ocr_options = RapidOcrOptions(backend="torch")

doc_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)
chunker = HybridChunker(max_tokens=512)
print("✅ Docling pipeline loaded successfully.")

## 🛠️ Step 4: Define API Schemas and Router

In [ ]:
app = FastAPI(title="Unified Kaggle AI API Server")

tasks_store = {}

# --- Pydantic Schemas ---
class ChatMessage(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    messages: list[ChatMessage]
    temperature: float = 0.0
    max_new_tokens: int = 512

class ChatCompletionResponse(BaseModel):
    success: bool
    response: str
    error_message: str

class EmbedRequest(BaseModel):
    texts: list[str]

class EmbedResponse(BaseModel):
    success: bool
    embeddings: list[list[float]]
    error_message: str

class RerankRequest(BaseModel):
    query: str
    documents: list[str]

class RerankResultItem(BaseModel):
    index: int
    score: float

class RerankResponse(BaseModel):
    success: bool
    results: list[RerankResultItem]
    error_message: str

# --- Processing Helpers ---
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

def get_embeddings_internal(texts: list[str]) -> list[list[float]]:
    import torch.nn.functional as F
    encoded_input = embed_tokenizer(texts, padding=True, truncation=True, return_tensors='pt').to(embed_model.device)
    with torch.no_grad():
        model_output = embed_model(**encoded_input)
    embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
    embeddings = F.normalize(embeddings, p=2, dim=1)
    return embeddings.tolist()

def generate_chat_internal(messages, temperature=0.0, max_new_tokens=512):
    text = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = llm_tokenizer(text, return_tensors="pt").to(llm_model.device)
    do_sample = temperature > 0.0
    gen_kwargs = {"max_new_tokens": max_new_tokens}
    if do_sample:
        gen_kwargs["do_sample"] = True
        gen_kwargs["temperature"] = temperature
    else:
        gen_kwargs["do_sample"] = False
        
    with torch.no_grad():
        outputs = llm_model.generate(**inputs, **gen_kwargs)
    response = llm_tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )
    return response

def compute_rerank_scores_internal(query: str, documents: list[str]) -> list[dict]:
    pairs = [[query, doc] for doc in documents]
    features = reranker_tokenizer(
        pairs,
        padding=True,
        truncation=True,
        return_tensors="pt"
    ).to(reranker_model.device)
    
    with torch.no_grad():
        outputs = reranker_model(**features)
        scores = outputs.logits.view(-1).float().cpu().tolist()
        
    results = [
        {"index": idx, "score": float(score)}
        for idx, score in enumerate(scores)
    ]
    results.sort(key=lambda x: x["score"], reverse=True)
    return results

def process_pdf_in_background(task_id: str, tmp_path: str, company: str, year: int):
    try:
        print(f"[{task_id}] Starting background parsing on GPU 1...")
        result = doc_converter.convert(tmp_path)
        doc = result.document

        chunk_list = list(chunker.chunk(dl_doc=doc))

        chunks = []
        for chunk in chunk_list:
            pages = sorted({
                prov.page_no 
                for item in chunk.meta.doc_items 
                for prov in item.prov 
                if hasattr(prov, "page_no")
            })
            page_number = pages[0] if pages else 1
            
            section_path = " > ".join(chunk.meta.headings) if chunk.meta.headings else "General"
            context_text = chunker.contextualize(chunk)
            
            chunks.append({
                "text": context_text,
                "page_number": page_number,
                "section": section_path
            })

        if not chunks:
            tasks_store[task_id] = {"status": "completed", "chunks": []}
            return

        print(f"[{task_id}] Generating embeddings for {len(chunks)} chunks on GPU 1...")
        texts_to_embed = [
            f"search_document: [Company: {company} | Year: {year}] {c['text']}"
            for c in chunks
        ]
        
        batch_size = 32
        all_embeddings = []
        for i in range(0, len(texts_to_embed), batch_size):
            batch = texts_to_embed[i:i+batch_size]
            all_embeddings.extend(get_embeddings_internal(batch))

        output_chunks = []
        for chunk, embedding in zip(chunks, all_embeddings):
            output_chunks.append({
                "text": chunk["text"],
                "page_number": chunk["page_number"],
                "section": chunk["section"],
                "embedding": embedding
            })

        tasks_store[task_id] = {
            "status": "completed",
            "chunks": output_chunks
        }
        print(f"[{task_id}] Background parsing completed successfully.")
    except Exception as e:
        import traceback
        err_msg = traceback.format_exc()
        print(f"[{task_id}] Error in background task:\n{err_msg}")
        tasks_store[task_id] = {
            "status": "failed",
            "error_message": str(e)
        }
    finally:
        import os
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

# --- HTTP REST Endpoints ---
@app.get("/")
def root():
    return {
        "status": "running",
        "llm": {"model": llm_model_name, "device": str(next(llm_model.parameters()).device)},
        "embeddings": {"model": embed_model_name, "device": str(embed_model.device)},
        "reranker": {"model": reranker_model_name, "device": str(reranker_model.device)}
    }

# LLM completions route
@app.post("/generate")
def generate(req: ChatCompletionRequest):
    try:
        result = generate_chat_internal(
            messages=[{"role": m.role, "content": m.content} for m in req.messages],
            temperature=req.temperature,
            max_new_tokens=req.max_new_tokens
        )
        return ChatCompletionResponse(success=True, response=result, error_message="")
    except Exception as e:
        return ChatCompletionResponse(success=False, response="", error_message=str(e))

# Vector Embeddings route
@app.post("/embed")
def embed(req: EmbedRequest):
    try:
        embeddings = get_embeddings_internal(req.texts)
        return EmbedResponse(success=True, embeddings=embeddings, error_message="")
    except Exception as e:
        return EmbedResponse(success=False, embeddings=[], error_message=str(e))

# Cross-Encoder Reranker route
@app.post("/rerank")
def rerank(req: RerankRequest):
    try:
        results = compute_rerank_scores_internal(req.query, req.documents)
        return RerankResponse(
            success=True,
            results=[RerankResultItem(**item) for item in results],
            error_message=""
        )
    except Exception as e:
        return RerankResponse(success=False, results=[], error_message=str(e))

# PDF Async Parser route
@app.post("/parse-pdf-async")
async def parse_pdf_async(
    background_tasks: BackgroundTasks,
    file: UploadFile = File(...),
    company: str = Form(...),
    year: int = Form(...)
):
    try:
        task_id = str(uuid.uuid4())
        tasks_store[task_id] = {"status": "processing"}
        
        with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as tmp:
            content = await file.read()
            tmp.write(content)
            tmp_path = tmp.name

        background_tasks.add_task(
            process_pdf_in_background,
            task_id,
            tmp_path,
            company,
            year
        )
        
        return {"success": True, "task_id": task_id}
    except Exception as e:
        return {"success": False, "task_id": "", "error_message": str(e)}

# PDF Parser Status route
@app.get("/parse-status/{task_id}")
def get_parse_status(task_id: str):
    status = tasks_store.get(task_id)
    if not status:
        return {"status": "not_found"}
    return status

## 🚀 Step 5: Start Public Tunnel and FastAPI Server

In [ ]:
# Apply nesting patch to allow uvicorn inside Jupyter event loop
nest_asyncio.apply()

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(PORT)
    print("\n" + "="*80)
    print(f"🚀 SERVER PUBLIC URL: {public_url}")
    print("Copy the URL above and paste it into your Backed/.env as:")
    print("  LLM_SERVER_URL, EMBEDDING_SERVER_URL, and RERANKER_SERVER_URL")
    print("="*80 + "\n")
else:
    print("\n❌ Unable to set up ngrok tunnel without an authentication token.")

# Launch server directly in the notebook's active event loop
import uvicorn
config = uvicorn.Config(app, host="0.0.0.0", port=PORT, loop="asyncio")
server = uvicorn.Server(config)
await server.serve()